In [1]:
!python -m pip install transformers datasets accelerate sentencepiece
!pip install --upgrade transformers
#28s

In [2]:
from google.colab import files
uploaded = files.upload()

Saving test.json to test.json
Saving train.json to train.json
Saving valid.json to valid.json


In [3]:
from datasets import Dataset
import torch
from transformers import DataCollatorForSeq2Seq, AutoTokenizer, AutoModelForSeq2SeqLM,Seq2SeqTrainingArguments, Seq2SeqTrainer,  EarlyStoppingCallback

#41s

# Load model gốc

In [ ]:
model_checkpoint = "vietai/vit5-base" 
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
#23s

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

# Load dataset

In [8]:
from datasets import load_dataset
split_datasets = load_dataset(
    "json",
    data_files={
        "train": "train.json",
        "validation": "valid.json",
        "test": "test.json",
    }
)
print(split_datasets)


DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 11379
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 1422
    })
    test: Dataset({
        features: ['input', 'target'],
        num_rows: 1423
    })
})


In [ ]:
def preprocess_function(examples):
    inputs = examples["input"]
    targets = examples["target"]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names
)

Map:   0%|          | 0/11379 [00:00<?, ? examples/s]

Map:   0%|          | 0/1422 [00:00<?, ? examples/s]

Map:   0%|          | 0/1423 [00:00<?, ? examples/s]

# Khởi tạo quá trình train

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def get_trainer(model, tokenizer, n_epochs):
  args = Seq2SeqTrainingArguments(
      output_dir="./vnec_model",
      eval_strategy="epoch",           
      learning_rate=1e-4,
      per_device_train_batch_size=16,
      per_device_eval_batch_size=16,
      weight_decay=0.01,
      save_total_limit=3,
      num_train_epochs=n_epochs,
      predict_with_generate=True,
      fp16=True,                       # Bật nếu dùng GPU T4 trên Colab
      logging_dir='./logs',
      warmup_ratio=0.1,
      save_strategy="epoch",

      load_best_model_at_end=True,
      metric_for_best_model="eval_loss",
      greater_is_better=False,
  )

  trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,    
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
  )
  return trainer


# Train model với 1 epoch

In [17]:
trainer = get_trainer(model, tokenizer, 1)
trainer.train(resume_from_checkpoint=False)
trainer.save_model("./trained_1_epoch/")
#Note: Enter 3 for wandb

/tmp/ipython-input-3671242433.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,0.496500,1.144854


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


# Load model

In [ ]:
import os
import glob
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def load_model(base_path):
    if os.path.exists(os.path.join(base_path, "config.json")):
        print("Tìm thấy model ở thư mục gốc. Đang load...")
        final_path = base_path
    else:
        checkpoints = glob.glob(os.path.join(base_path, "checkpoint-*"))
        if len(checkpoints) > 0:
            final_path = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
            print(f"Không thấy model gốc, đang load checkpoint mới nhất: {final_path}")
        else:
            raise ValueError("Không tìm thấy model hay checkpoint nào cả!")

    tokenizer = AutoTokenizer.from_pretrained(final_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(final_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    return model, tokenizer

# Train tiếp 13 epoch

In [33]:
base_path = "./trained_1_epoch"
model, tokenizer = load_model(base_path)

Tìm thấy model ở thư mục gốc. Đang load...


In [35]:
trainer = get_trainer(model, tokenizer, 13)
trainer.train(resume_from_checkpoint=False)
trainer.save_model("./trained")

/tmp/ipython-input-3421856439.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,0.094100,1.225317
2,0.201200,1.312916
3,0.195500,1.350957


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


# Chạy thử model

In [36]:
base_path = "./trained"
model, tokenizer = load_model(base_path)

Tìm thấy model ở thư mục gốc. Đang load...


In [ ]:
def correct_sentence(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True)
    inputs = inputs.to(device)

    with torch.no_grad(): 
        output_ids = model.generate(
            **inputs,
            max_length=128,
            num_beams=5, 
            early_stopping=True
        )

    # Decode ra văn bản
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
model.to(device)
input_text = "hom noy tròi thât dep"
corrected_text = correct_sentence(input_text)

print("-" * 30)
print(f"Input:  {input_text}")
print(f"Output: {corrected_text}")
print("-" * 30)

------------------------------
Input:  hom noy tròi thât dep
Output: hom ny trời thât đẹp
------------------------------


# Ultilities

In [38]:
import shutil

folder_name = "trained"
source = f'/content/{folder_name}'
destination = f'/content/drive/MyDrive/{folder_name}'

try:
    from google.colab import drive
    drive.mount('/content/drive')

    shutil.copytree(source, destination)
    print("Copy thành công!")
except FileExistsError:
    print("Thư mục đích đã tồn tại trên Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copy thành công!
